In [ ]:
import json
import os
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rctoolbox.utils.project_root import find_project_root

project_root = find_project_root()
os.chdir(project_root)
print(f"Current working directory: {os.getcwd()}")

def reset_logging():
    import logging
    logging.root.handlers.clear()
    for logger in logging.Logger.manager.loggerDict.values():
        if hasattr(logger, "handlers"):
            logger.handlers.clear()

In [ ]:
HEMS_ID = "AIBF"

OUT_DIR_PATH = project_root / "examples" / "run_identification" / "five_r1c_eniso13790" / HEMS_ID
# print(f"Output dir: {OUT_DIR_PATH}")

## DATA

In [ ]:
DATA_DIR  = project_root / "examples" / "data"

data_path = DATA_DIR / f'{HEMS_ID}_data.pkl'
DATA = pd.read_pickle(data_path)
SEASON = 'w2324'
season = DATA[SEASON]
df = season['df']

print('season:        ', SEASON)
print('df shape:      ', df.shape)
print('time range:    ', df.index.min(), '->', df.index.max())

In [ ]:
# Define time windows for training

from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

wdh = 24; spb = int(1e10); ssh=1;
pre_window_lag = timedelta(hours=24)

t1_train = datetime(2023,12,1)
t2_train = datetime(2023,12,22)
tws_train = generate_time_windows(start_time=t1_train, end_time=t2_train, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

t1_val = datetime(2023,12,22)
t2_val = datetime(2024,1,1)
tws_val = generate_time_windows(start_time=t1_val, end_time=t2_val, window_duration_hours=wdh, samples_per_batch=spb, slide_step_hours=ssh)

tws = {"train": tws_train, "val": tws_val}
dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [ ]:
RESULTS_DIR_PATH = OUT_DIR_PATH / "results"
OUT_DIR_PATH.mkdir(parents=True, exist_ok=True)
RESULTS_DIR_PATH.mkdir(exist_ok=True)

from rctoolbox.models.rc.five_r1c_eniso13790.build_data_dict import build_data_dict
import joblib
data_joblib = {
    "train": build_data_dict(dataloader, "train"),
    "val":   build_data_dict(dataloader, "val"),
}
data_joblib_path = OUT_DIR_PATH / "data_train.joblib"
joblib.dump(data_joblib, data_joblib_path)

## Identification
---

In [ ]:
from rctoolbox.utils.config_loader import load_default_building_config
building_config = load_default_building_config(HEMS_ID)
building_config

In [ ]:
metric = "MAE"
model_config        = {"model_type": "5R1C_heat_demand", "init_config": {"method": "regime_aware", "variant": "last"}}
optimization_config = {"objective_function": metric, "std_weight": 0.0, "pop_size": 100, "n_gen": 150}
selection_config    = {"criteria": "val error", "select_R_star":True, "threshold_config": {"type": "idx", "x": 13000}}
clustering_config   = {"min_samples_ratio": 0.1, "cl_eval_metric": metric, "cl_eval_criteria": "val error"}

identification_config = {"model_config": model_config, 
                      "optimization_config": optimization_config, 
                      "selection_config": selection_config, 
                      "clustering_config": clustering_config}

identification_config_path  = OUT_DIR_PATH / "identification_config.json"
with open(identification_config_path, "w") as f:
    json.dump(identification_config, f, indent=4)

In [ ]:
# --- RUN PIPELINE ---
reset_logging()

from argparse import Namespace
from rctoolbox.pipeline.runner import run_pipeline

run_pipeline(
    Namespace(
        building_config=HEMS_ID,
        data_path=str(data_joblib_path),
        identification_config=str(identification_config_path),
        results_directory_path=str(RESULTS_DIR_PATH),
        seed=23, save_figures=True, save_optimizer=False, save_dfRtopN=False, stage="1-3"
    )
)

## Evaluate
---

In [ ]:
# Load the identification results.
results_path = f"{OUT_DIR_PATH}/results/results.json"
with open(results_path, "r") as f:
    results = json.load(f)
results

In [ ]:
# Set up the model
from rctoolbox.models.rc.five_r1c_eniso13790.model import FiveR1C_ENISO13790
from rctoolbox.models.loader import load_best_model
model = load_best_model(results_path=results_path, model_cls=FiveR1C_ENISO13790, best_from_cluster=0)
model.params

In [ ]:
## GENERATE TIME WINDOWS

from rctoolbox.data.time_windows_generator import generate_time_windows
from rctoolbox.data.numpydataloader import DataLoader as NumpyDataLoader

t1 = datetime(2023,12,1)
t2 = datetime(2024,4,1)

wdh = 24; ssh = wdh
tws = generate_time_windows(start_time=t1, end_time=t2, samples_per_batch=int(1e6), window_duration_hours=wdh, slide_step_hours=ssh)
tws_test = {"test": tws}

dataloader = NumpyDataLoader(df)
dataloader.load_all_data_batches(time_windows=tws_test, features=df.columns.tolist(), pre_window_lag=pre_window_lag)

In [ ]:
# RUN SIMULATION on the test batch.
from rctoolbox.models.rc.five_r1c_eniso13790.predict_functions import predict_for_batch, predict_for_batch_and_evaluate
from rctoolbox.evaluation.forecast_metrics import evaluate_forecast
from rctoolbox.evaluation.error_threshold import err_thr

batch = dataloader.batches["test"]["batch_0"]
raw_dfs = dataloader.raw_dfs["test"]["batch_0"]

# Forecast
_, batch_dfs = predict_for_batch(model=model, batch=batch,  y1_set_col="y1_set", raw_dfs=raw_dfs, use_control_day_mask = False)
dfbs = pd.concat([df_ for df_ in batch_dfs.values()], axis=0)

In [ ]:
# PLOT
fig, ax = plt.subplots(figsize=(15, 5))
axr = ax.twinx()
ds = "default"

t1 = datetime(2024, 3, 1)
t2 = t1 + timedelta(days=30)
dfbs_ = dfbs.loc[t1:t2]

# Right axis: setpoint and measured indoor temperature
dfbs_["y1_set"].plot(
    ax=axr, color="blue", lw=1, ls="--", alpha=1, drawstyle=ds,
    label=r"Setpoint  $y1\_set$",
)
dfbs_["y1"].plot(
    ax=axr, color="tab:blue", lw=0.5, alpha=0.5,
    label=r"Indoor temp  $y1$",
)

# Left axis: measured and predicted heat power
dfbs_["Phi_HC"].plot(
    ax=ax, color="k", lw=2, alpha=1, drawstyle=ds,
    label=r"Measured  $\Phi_{HC}$",
)
dfbs_["Phi_HC_pred"].plot(
    ax=ax, color="red", lw=2, ls="--", alpha=1, drawstyle=ds,
    label=r"Predicted  $\hat{\Phi}_{HC}$",
)

# Axis labels
ax.set_ylabel("Heat power  [W]", fontsize=15)
axr.set_ylabel("Temperature  [°C]", fontsize=15)
ax.set_xlabel("")

# Combined legend from both axes
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = axr.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="lower right", fontsize=12, ncol=2)

plt.tight_layout()

# Save
fig_out_path = project_root / "examples/figures/five_r1c_iso_forecast.pdf"
fig.savefig(fig_out_path, bbox_inches="tight")
fig.savefig(fig_out_path.with_suffix(".png"), bbox_inches="tight", dpi=200)
# print(f"Saved: {fig_out_path}")
# print(f"Saved: {fig_out_path.with_suffix('.png')}")

In [ ]:
# Predict and evaluate
p = predict_for_batch_and_evaluate(model, batch, y1_set_col="y1_set", err_thr=err_thr, use_control_day_mask = True)

from rctoolbox.evaluation.forecast_metrics import organise_results_in_dataframe
summary_df = organise_results_in_dataframe(p['summary_statistics'])
summary_df